In [25]:
# Create a new cell in your notebook 
import pandas as pd 
import os 
import folium #import folium for crseate maps
from folium import plugins 

#set workimg directory 
os.chdir(r'D:\Mission_Blitzkreig')

# Load your campaign data 
df = pd.read_csv(r'D:\Mission_Blitzkreig\Month_1\ooh_campaigns.csv')

# Calculate Metrics 
df['roi'] = ((df['revenue'] - df['spend']) / df['spend'] * 100).round(2)  # Calculate ROI percentage
df['ctr'] = (df['clicks'] / df['impressions'] * 100).round(2)  # Calculate CTR percentage
df['profit'] = df['revenue'] - df['spend']  # Calculate profit in euros

print("✅ Data loaded!")
print(f"Total campaigns: {len(df)}")


✅ Data loaded!
Total campaigns: 10


In [26]:
# Add Geolocation
billboard_locations = pd.DataFrame ({
    'campaign_name' :  df['campaign_name'],
    'city' : df['city'],
    'format' : df['format'],
    'revenue' :df['revenue'],
    'roi' : df['roi'],
    'latitude' :[
        53.3498,  # Dublin City Centre
        53.3480,  # Dublin (another location)
        51.8985,  # Cork
        51.9000,  # Cork (another location)
        54.5973,  # Belfast
        54.6000,  # Belfast (another location)
        53.2707,  # Galway
        52.6638,  # Limerick
        52.6700,  # Limerick (another location)
        52.2593   # Waterford
    ],
    
    'longitude': [
        -6.2603,   # Dublin City Centre
        -6.2500,   # Dublin
        -8.4756,   # Cork
        -8.4600,   # Cork
        -5.9301,   # Belfast
        -5.9400,   # Belfast
        -9.0568,   # Galway
        -8.6267,   # Limerick
        -8.6100,   # Limerick
        -7.1101    # Waterford
    ]
})

print ("\n Billboard location created!")
print (billboard_locations.head()) 


 Billboard location created!
                   campaign_name     city       format  revenue     roi  \
0  Dublin City Centre - 48 Sheet   Dublin     48 Sheet    18900  350.00   
1        Cork Motorway - Digital     Cork      Digital    11200  300.00   
2     Belfast Roadside - 6 Sheet  Belfast      6 Sheet     5600  273.33   
3     Dublin Airport - Supersite   Dublin    Supersite    48000  464.71   
4      Galway City - Bus Shelter   Galway  Bus Shelter     2800  194.74   

   latitude  longitude  
0   53.3498    -6.2603  
1   53.3480    -6.2500  
2   51.8985    -8.4756  
3   51.9000    -8.4600  
4   54.5973    -5.9301  


In [27]:
# Part 4  : Create the map 
ireland_map = folium.Map(
    location=[53.4, -7.9],
    zoom_start=7,
    titles='OpenStreetMap'
)
print("Base Map Created!")

Base Map Created!


In [28]:
# Part 5 : Add Markers to the map 
for idx, row in billboard_locations.iterrows():

    if row['roi'] > 100:
        color ='green' # If roi is good 
    elif row['row'] > 50:
        color = 'orange' # if roi is average 
    else:
        color = 'red' # if roi is not good 

    popup_text = f""" 
<b>{row['campaign_name']}</b><br>

City: {row['city']}<br>
Format:{row}<br>
Revenue:€{row['revenue']:,.0f}<br>
ROI:{row['roi']:.1f}%
"""
    
# Add marker to the map 
folium.Marker(
    location=[row['latitude'], row['longitude']],
    popup=folium.Popup(popup_text, max_width=250),
    icon=folium.Icon(color=color, icon='info-sign')

).add_to(ireland_map)
print(f"Added {len(billboard_locations)} markers to map!")


Added 10 markers to map!


In [29]:
# part 6 : Save & View map 
ireland_map.save('billboard_locations_map.html')

print("✅ Interactive map created: billboard_locations_map.html")
print("\n📍 To view the map:")
print("1. Go to D:\\Mission_Blitzkreig\\")
print("2. Double-click 'billboard_locations_map.html'")
print("3. It will open in your browser!")
print("\n🎨 Marker colors:")
print("  🟢 Green = ROI > 100% (highly profitable)")
print("  🟠 Orange = ROI 50-100% (moderately profitable)")
print("  🔴 Red = ROI < 50% (needs improvement)")


✅ Interactive map created: billboard_locations_map.html

📍 To view the map:
1. Go to D:\Mission_Blitzkreig\
2. Double-click 'billboard_locations_map.html'
3. It will open in your browser!

🎨 Marker colors:
  🟢 Green = ROI > 100% (highly profitable)
  🟠 Orange = ROI 50-100% (moderately profitable)
  🔴 Red = ROI < 50% (needs improvement)


In [30]:
# Part 7 : Add a legend to the map 
ireland_map_with_legend = folium.Map(
    location=[53.4, -7.9],
    zoom_start=7,
    titles='OpenStreetMap'
)


# Add all markers 
for idx, row in billboard_locations.iterrows():
    if row['roi'] > 100:
        color ='green'
    elif row['roi'] > 50:
        color = 'orange'
    else:
        color = 'red'

    popup_text = f"""
    <b>{row['campaign_name']}</b><br>
    City: {row['city']}<br>
    Format: {row['format']}<br>
    Revenue: €{row['revenue']:,.0f}<br>
    ROI: {row['roi']:.1f}%
    """
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=folium.Popup(popup_text, max_width=250),
        icon=folium.Icon(color=color, icon='info-sign')
    ).add_to(ireland_map_with_legend)

    # Add custom legend HTML
# This creates a legend box in the top-right corner
legend_html = '''
<div style="position: fixed; 
            top: 10px; right: 10px; width: 200px; height: 120px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px">
<!-- CSS styling for the legend box -->

<p><strong>Billboard Performance</strong></p>
<!-- Title of the legend -->

<p><span style="color: green;">●</span> High ROI (&gt;100%)</p>
<!-- Green circle for high ROI -->

<p><span style="color: orange;">●</span> Medium ROI (50-100%)</p>
<!-- Orange circle for medium ROI -->

<p><span style="color: red;">●</span> Low ROI (&lt;50%)</p>
<!-- Red circle for low ROI -->

</div>
'''

# Add legend to map
ireland_map_with_legend.get_root().html.add_child(folium.Element(legend_html))
# .get_root() gets the map's HTML structure
# .html.add_child() adds our legend HTML to it

# Save map with legend
ireland_map_with_legend.save('billboard_map_with_legend.html')
print("✅ Map with legend saved: billboard_map_with_legend.html")
    

✅ Map with legend saved: billboard_map_with_legend.html
